# 🎯 YOLO + SAHI + Crop Classifier → Visual Search Engine — Tuần 4
## Luồng dữ liệu theo đề xuất của Thầy: Detection → Crop → Embed → FAISS Retrieval
### 👤 Mã Gia Vỹ | Nhóm 3

---

## 🗺️ Luồng dữ liệu tổng thể

```
Input Image
    │
    ▼
┌─────────────────────────────────┐
│  YOLOv8n / YOLOv11n Detector    │  ← Phát hiện vùng sản phẩm
│  conf_threshold = 0.15 (thấp)   │    Hạ threshold để tăng recall
└──────────────┬──────────────────┘    (Khắc phục: Error Cascade)
               │
               ▼
┌─────────────────────────────────┐
│  SAHI Adaptive Tiling           │  ← Tăng cường vật thể nhỏ
│  (Batch Inference)              │    Batch tất cả tiles → 1 pass
└──────────────┬──────────────────┘    (Khắc phục: SAHI Latency)
               │
               ▼
         Crop Detected Boxes
               │
               ▼
┌─────────────────────────────────┐
│  EfficientNetB0 Backbone        │  ← Encoder 2 đầu ra:
│  ├── [Head 1] Softmax → label   │    1. Classification (label + conf)
│  └── [Head 2] Embed 512-dim     │    2. Embedding (retrieval)
│  + Temperature Scaling          │    (Khắc phục: Confidence Calibration)
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  Soft Fusion (calibrated)       │  ← α[class] × YOLO_conf
│  α per-class (learned on val)   │    + β[class] × Classifier_conf
└──────────────┬──────────────────┘    (Khắc phục: Fusion Cứng nhắc)
               │
               ▼
┌─────────────────────────────────┐
│  FAISS IndexFlatIP              │  ← Cosine similarity search
│  Gallery: 34,250 embeddings     │    Top-K similar products
│  + pHash Boost + AQE            │    (Khắc phục: Thiếu Retrieval)
└──────────────┬──────────────────┘
               │
               ▼
   Final: Label + BBox + Confidence + Top-5 Similar
```

---

| Nhược điểm | Giải pháp áp dụng |
|---|---|
| ⚡ SAHI Latency | Batch inference tất cả tiles → 1 GPU forward pass |
| 🔗 Error Cascade | YOLO conf_threshold=0.15, classifier làm bộ lọc thứ 2 |
| 📏 Fusion cứng nhắc | Temperature Scaling + α[class] học trên val set |
| 🔍 Thiếu Retrieval | Penultimate embedding 512-dim → FAISS + pHash + AQE |

---

### 📋 Quy trình Notebook:
1. **Bước 0:** Cài đặt & Import
2. **Bước 1:** Phân chia Validation/Test Set (20%/80%) — KHÔNG DATA LEAKAGE
3. **Bước 2:** YOLO Detection + SAHI (Batch Tiling) → Crops Gallery
4. **Bước 3:** EfficientNetB0 Dual-Head (Classify + Embed) → 512-dim embeddings
5. **Bước 4:** Temperature Scaling + Per-class α Fusion (trên Val Set)
6. **Bước 5:** Build FAISS Index (Gallery Embeddings)
7. **Bước 6:** Đánh giá cuối cùng: mAP@5, Precision@1, Recall@5 + Export CSV

In [ ]:
# ============================================================
# 📦 BƯỚC 0.1: CÀI ĐẶT THƯ VIỆN CẦN THIẾT (Google Colab)
# ============================================================

import subprocess, sys

def install_if_missing(package, pip_name=None):
    """Cài đặt thư viện nếu chưa có, tránh cài lại không cần thiết."""
    pip_name = pip_name or package
    try:
        __import__(package)
        print(f'✅ {pip_name} đã được cài đặt sẵn.')
    except ImportError:
        print(f'🔄 Đang cài đặt {pip_name}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pip_name, '-q'],
                       capture_output=True)
        print(f'✅ Đã cài xong {pip_name}!')

# Cài đặt các thư viện cần thiết
install_if_missing('faiss', 'faiss-gpu')          # FAISS GPU/CPU
install_if_missing('imagehash', 'imagehash')       # pHash
install_if_missing('ultralytics', 'ultralytics')   # YOLOv8/v11
install_if_missing('sahi', 'sahi')                 # SAHI sliced inference
install_if_missing('timm', 'timm')                 # EfficientNet

# Thử cài faiss-gpu, nếu thất bại dùng faiss-cpu
try:
    import faiss
except ImportError:
    print('⚠️  faiss-gpu thất bại, đang cài faiss-cpu...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'faiss-cpu', '-q'])
    import faiss

print('\n🎉 Tất cả thư viện đã sẵn sàng!')

In [ ]:
# ============================================================
# 🔗 BƯỚC 0.2: KẾT NỐI GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Đã kết nối Google Drive thành công!')

In [ ]:
# ============================================================
# 📚 BƯỚC 0.3: IMPORT CÁC THƯ VIỆN
# ============================================================
import os
import re
import time
import shutil
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# YOLO & SAHI
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.postprocess.combine import NMMPostprocess

# Sklearn
from sklearn.model_selection import train_test_split

# FAISS
import faiss

# pHash
import imagehash

warnings.filterwarnings('ignore')

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print('🖥️  THÔNG TIN PHẦN CỨNG')
print('=' * 60)
print(f'Thiết bị PyTorch : {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Tên GPU          : {gpu_name}')
    print(f'VRAM             : {gpu_mem:.2f} GB')
    print(f'CUDA Version     : {torch.version.cuda}')
else:
    print('⚠️  KHÔNG CÓ GPU — Tốc độ xử lý sẽ rất chậm!')
print('=' * 60)

In [ ]:
# ============================================================
# ⚙️  BƯỚC 0.4: CẤU HÌNH DỰ ÁN — THAY ĐỔI ĐƯỜNG DẪN TẠI ĐÂY
# ============================================================

# --- Đường dẫn gốc trên Google Drive ---
BASE_DIR        = '/content/drive/MyDrive/DuLieuPython'

# --- Đường dẫn thư mục ảnh (chứa 34,250 ảnh Shopee) ---
IMAGE_DIR       = os.path.join(BASE_DIR, 'train_images')

# --- File CSV chính (34,250 dòng) ---
CANDIDATE_CSV   = os.path.join(BASE_DIR, 'train.csv')

# --- Thư mục lưu features đã trích xuất ---
PROCESSED_DIR   = os.path.join(BASE_DIR, 'processed_yolo_sahi')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Thư mục lưu kết quả ---
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Thư mục lưu checkpoint (chống Colab timeout) ---
CKPT_DIR        = os.path.join(BASE_DIR, 'checkpoints_yolo_sahi')
os.makedirs(CKPT_DIR, exist_ok=True)

# ============================================================
# ⚙️  CẤU HÌNH YOLO + SAHI
# ============================================================

# YOLOv8n: nhẹ nhất, phù hợp Colab T4
# Thay bằng 'yolov8s.pt' nếu muốn accuracy cao hơn
YOLO_MODEL_NAME = 'yolov8n.pt'

# QUAN TRỌNG: Hạ conf_threshold xuống 0.15 để tăng recall
# Classifier sẽ đóng vai trò bộ lọc thứ 2 — khắc phục Error Cascade
YOLO_CONF_THRESH = 0.15
YOLO_IOU_THRESH  = 0.45

# SAHI: Kích thước tile và overlap
# Với Shopee (ảnh sản phẩm cỡ 800-1200px) → tile 512 là hợp lý
SAHI_SLICE_H     = 512
SAHI_SLICE_W     = 512
SAHI_OVERLAP     = 0.1   # Giảm overlap để ít tiles hơn → nhanh hơn

# ============================================================
# ⚙️  CẤU HÌNH CLASSIFIER + EMBEDDING
# ============================================================

# EfficientNetB0: 5.3M params, phù hợp T4
EMBED_DIM        = 512   # Chiều embedding (penultimate layer)
BATCH_SIZE_EMBED = 64    # Batch inference cho embedding
IMG_SIZE_CROP    = 224   # Resize crop trước khi embed

# ============================================================
# ⚙️  CẤU HÌNH RETRIEVAL
# ============================================================

TOP_K           = 5      # Số kết quả trả về
TOP_RERANK      = 50     # Số ứng viên ban đầu từ FAISS trước pHash

# pHash Boost — giữ giống với notebook SigLIP
PHASH_THRESHOLD = 5
PHASH_BOOST     = 0.15

# AQE (Average Query Expansion)
AQE_K           = 3

# Temperature Scaling ban đầu (sẽ học trên val set)
INIT_TEMPERATURE = 1.5

print('✅ Cấu hình đã sẵn sàng!')
print(f'   BASE_DIR         : {BASE_DIR}')
print(f'   IMAGE_DIR        : {IMAGE_DIR}')
print(f'   CANDIDATE_CSV    : {CANDIDATE_CSV}')
print(f'   YOLO Model       : {YOLO_MODEL_NAME}')
print(f'   YOLO Conf        : {YOLO_CONF_THRESH}  (thấp để tăng recall)')
print(f'   SAHI Tile        : {SAHI_SLICE_H}×{SAHI_SLICE_W}, overlap={SAHI_OVERLAP}')
print(f'   Embed Dim        : {EMBED_DIM}')
print(f'   Batch Embed      : {BATCH_SIZE_EMBED}')
print(f'   TOP_K            : {TOP_K}  |  TOP_RERANK: {TOP_RERANK}')

In [ ]:
# ============================================================
# 🛠️  CÁC HÀM TIỆN ÍCH (HELPER FUNCTIONS)
# Giữ nguyên interface giống notebook SigLIP để tương thích
# ============================================================

def l2_normalize(features: np.ndarray) -> np.ndarray:
    """
    Chuẩn hóa L2 cho ma trận features.
    Mỗi hàng sẽ có norm = 1 sau khi chuẩn hóa.
    """
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    # Tránh chia cho 0
    norms = np.where(norms < 1e-10, 1e-10, norms)
    return (features / norms).astype(np.float32)


def hex_to_phash_array(hex_str) -> np.ndarray:
    """
    Chuyển chuỗi hex của pHash thành mảng bool (64 bits).
    Dùng cho tính khoảng cách Hamming vectorized.
    """
    try:
        phash_obj = imagehash.hex_to_hash(str(hex_str))
        return phash_obj.hash.flatten()  # (64,) bool array
    except Exception:
        return np.zeros(64, dtype=bool)


def compute_hamming_distances(query_hash: np.ndarray,
                               candidate_hashes: np.ndarray) -> np.ndarray:
    """
    Tính khoảng cách Hamming vectorized giữa 1 query và nhiều candidates.

    Args:
        query_hash      : (64,) bool array — pHash của query
        candidate_hashes: (N, 64) bool array — pHash của N candidates

    Returns:
        distances: (N,) int32 array — khoảng cách Hamming
    """
    return np.sum(query_hash != candidate_hashes, axis=1).astype(np.int32)


def compute_map_at_k(queries_df: pd.DataFrame,
                     gallery_df: pd.DataFrame,
                     top_indices: np.ndarray,
                     k: int = 5) -> float:
    """
    Tính Mean Average Precision tại k (mAP@k) từ kết quả FAISS top-N.

    Args:
        queries_df  : DataFrame query (cột: posting_id, label_group)
        gallery_df  : DataFrame gallery (cột: posting_id, label_group)
        top_indices : (n_queries, N) — chỉ số gallery trả về bởi FAISS
        k           : số kết quả xem xét

    Returns:
        map_score: float — giá trị mAP@k
    """
    gallery_pids    = gallery_df['posting_id'].values
    gallery_labels  = gallery_df['label_group'].values

    # Tiền tính: label_group -> số lượng trong gallery
    label_counts = gallery_df['label_group'].value_counts().to_dict()

    ap_scores = []
    queries_iter = queries_df.reset_index(drop=True)

    for q_idx in range(len(queries_iter)):
        q_pid   = queries_iter.at[q_idx, 'posting_id']
        q_label = queries_iter.at[q_idx, 'label_group']

        # Số relevant items trong gallery (trừ chính query)
        n_relevant = label_counts.get(q_label, 0) - 1
        # Bỏ qua query không có ảnh liên quan trong gallery
        if n_relevant <= 0:
            continue

        # Lấy top-k kết quả, loại bỏ self-match
        retrieved_labels = []
        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:
                retrieved_labels.append(gallery_labels[gidx] == q_label)
            if len(retrieved_labels) == k:
                break

        # Tính Average Precision
        hits, precision_sum = 0, 0.0
        for rank, is_relevant in enumerate(retrieved_labels):
            if is_relevant:
                hits += 1
                precision_sum += hits / (rank + 1)

        ap = precision_sum / min(n_relevant, k)
        ap_scores.append(ap)

    return float(np.mean(ap_scores)) if ap_scores else 0.0


def compute_precision_at_1(queries_df: pd.DataFrame,
                            gallery_df: pd.DataFrame,
                            top_indices: np.ndarray) -> float:
    """
    Tính Precision@1: Tỷ lệ query có kết quả đầu tiên đúng nhãn.
    """
    gallery_pids   = gallery_df['posting_id'].values
    gallery_labels = gallery_df['label_group'].values
    queries_iter   = queries_df.reset_index(drop=True)

    correct = 0
    for q_idx in range(len(queries_iter)):
        q_pid   = queries_iter.at[q_idx, 'posting_id']
        q_label = queries_iter.at[q_idx, 'label_group']

        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:  # Bỏ self-match
                if gallery_labels[gidx] == q_label:
                    correct += 1
                break

    return correct / len(queries_iter)


def compute_recall_at_k(queries_df: pd.DataFrame,
                         gallery_df: pd.DataFrame,
                         top_indices: np.ndarray,
                         k: int = 5) -> float:
    """
    Tính Recall@k: Tỷ lệ trung bình relevant items được tìm thấy trong top-k.
    """
    gallery_pids   = gallery_df['posting_id'].values
    gallery_labels = gallery_df['label_group'].values
    label_counts   = gallery_df['label_group'].value_counts().to_dict()
    queries_iter   = queries_df.reset_index(drop=True)

    recall_scores = []
    for q_idx in range(len(queries_iter)):
        q_pid      = queries_iter.at[q_idx, 'posting_id']
        q_label    = queries_iter.at[q_idx, 'label_group']
        n_relevant = label_counts.get(q_label, 0) - 1
        if n_relevant <= 0:
            continue

        hits = 0
        count = 0
        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:
                if gallery_labels[gidx] == q_label:
                    hits += 1
                count += 1
            if count == k:
                break

        recall = hits / min(n_relevant, k)
        recall_scores.append(recall)

    return float(np.mean(recall_scores)) if recall_scores else 0.0


def build_faiss_index(gallery_features: np.ndarray,
                       use_gpu: bool = False,
                       gpu_resources=None) -> faiss.Index:
    """
    Xây dựng FAISS IndexFlatIP từ gallery features đã L2-normalize.

    Args:
        gallery_features: (N, D) float32 array đã chuẩn hóa L2
        use_gpu         : Có dùng GPU FAISS không
        gpu_resources   : faiss.StandardGpuResources nếu dùng GPU

    Returns:
        index: FAISS index đã thêm gallery
    """
    dim   = gallery_features.shape[1]
    index = faiss.IndexFlatIP(dim)
    if use_gpu and gpu_resources is not None:
        index = faiss.index_cpu_to_gpu(gpu_resources, 0, index)
    index.add(gallery_features.astype(np.float32))
    return index


def save_checkpoint(obj, name: str, ckpt_dir: str):
    """Lưu checkpoint vào Drive để chống Colab timeout."""
    path = os.path.join(ckpt_dir, name)
    if isinstance(obj, np.ndarray):
        np.save(path + '.npy', obj)
        print(f'💾 Checkpoint saved: {path}.npy')
    else:
        torch.save(obj, path + '.pt')
        print(f'💾 Checkpoint saved: {path}.pt')


print('✅ Đã định nghĩa xong tất cả hàm tiện ích!')
print('   - l2_normalize()              : Chuẩn hóa L2 vectorized')
print('   - hex_to_phash_array()        : Chuyển pHash hex → bool array')
print('   - compute_hamming_distances() : Khoảng cách Hamming vectorized')
print('   - compute_map_at_k()          : Tính mAP@k từ FAISS results')
print('   - compute_precision_at_1()    : Tính Precision@1')
print('   - compute_recall_at_k()       : Tính Recall@k')
print('   - build_faiss_index()         : Xây FAISS IndexFlatIP')
print('   - save_checkpoint()           : Lưu checkpoint chống timeout')

---
## 📊 Bước 1: Phân chia Validation / Test Set
- **Gallery:** Toàn bộ 34,250 ảnh (không thay đổi)
- **Val Set (20%):** Dùng để tune Temperature Scaling + α per-class
- **Test Set (80%):** Chỉ dùng một lần duy nhất để báo cáo kết quả cuối
- **Stratify:** Theo `label_group` để đảm bảo phân phối đồng đều

In [ ]:
# ============================================================
# 📊 BƯỚC 1: PHÂN CHIA DỮ LIỆU VALIDATION / TEST
# ============================================================
print('=' * 60)
print('BƯỚC 1: PHÂN CHIA DỮ LIỆU VALIDATION / TEST')
print('=' * 60)

# --- 1.1: Tải file CSV chính ---
print(f'\n📂 Đang tải dữ liệu từ: {CANDIDATE_CSV}')
candidate_df = pd.read_csv(CANDIDATE_CSV)

print(f'✅ Tải xong! Tổng số dòng: {len(candidate_df):,}')
print(f'   Các cột: {list(candidate_df.columns)}')

# Kiểm tra các cột bắt buộc
required_cols = ['posting_id', 'image', 'label_group']
missing_cols = [c for c in required_cols if c not in candidate_df.columns]
if missing_cols:
    raise ValueError(f'❌ Thiếu cột bắt buộc: {missing_cols}')
print(f'   Kiểm tra cột bắt buộc: ✅ OK')

# Thống kê cơ bản
unique_labels = candidate_df['label_group'].nunique()
avg_per_group = len(candidate_df) / unique_labels
print(f'\n📊 Thống kê dữ liệu:')
print(f'   Tổng số ảnh       : {len(candidate_df):,}')
print(f'   Số label_group    : {unique_labels:,}')
print(f'   TB ảnh/group      : {avg_per_group:.2f}')
print(f'   Max ảnh/group     : {candidate_df["label_group"].value_counts().max()}')
print(f'   Min ảnh/group     : {candidate_df["label_group"].value_counts().min()}')

print('\n   Mẫu dữ liệu (5 dòng đầu):')
display(candidate_df[required_cols + (['image_phash'] if 'image_phash' in candidate_df.columns else [])].head())

# --- 1.2: Phân chia Validation / Test (stratify theo label_group) ---
print('\n✂️  Đang phân chia dữ liệu (stratify theo label_group)...')
print('   - Val Set  : 20% (dùng để tune Temperature + α per-class)')
print('   - Test Set : 80% (CHỈ DÙNG ĐỂ ĐÁNH GIÁ CUỐI CÙNG)')

val_query_df, test_query_df = train_test_split(
    candidate_df,
    test_size=0.8,
    random_state=42,
    stratify=candidate_df['label_group'].values
)

# Reset index để tránh lỗi khi dùng .at[]
val_query_df  = val_query_df.reset_index(drop=True)
test_query_df = test_query_df.reset_index(drop=True)

# --- 1.3: Lưu kết quả phân chia ---
val_csv_path  = os.path.join(RESULTS_DIR, 'val_query_yolo.csv')
test_csv_path = os.path.join(RESULTS_DIR, 'test_query_yolo.csv')
val_query_df.to_csv(val_csv_path, index=False)
test_query_df.to_csv(test_csv_path, index=False)

# --- 1.4: In kết quả ---
print(f'\n✅ PHÂN CHIA HOÀN TẤT!')
print(f'   📁 Gallery (toàn bộ)  : {len(candidate_df):,} ảnh  (100%)')
print(f'   📁 Validation Set     : {len(val_query_df):,} ảnh  (20%)')
print(f'   📁 Test Set           : {len(test_query_df):,} ảnh  (80%)')

print(f'\n⚠️  NHẮC NHỞ QUAN TRỌNG:')
print(f'   ✅ Bước 4 (Tune Temperature + α per-class) → CHỈ DÙNG val_query_yolo.csv')
print(f'   ✅ Bước 6 (Đánh giá cuối) → CHỈ DÙNG test_query_yolo.csv — TUYỆT ĐỐI KHÔNG TUNE!')

---
## 🔍 Bước 2: YOLO Detection + SAHI Tiling → Crop Gallery

### Cải tiến so với pipeline gốc:
1. **Conf threshold = 0.15** (thấp) → tăng recall, classifier sẽ lọc lại
2. **Batch SAHI inference** → tất cả tiles của 1 ảnh gom vào 1 forward pass
3. **NMM post-processing** (Non-Maximum Merging) thay vì NMS → merge boxes thay vì xóa
4. **Cache crops vào Drive** → chống mất dữ liệu khi Colab timeout

### Lưu ý với Shopee dataset:
- Ảnh sản phẩm Shopee thường **đã được crop sẵn** (sản phẩm chiếm >70% diện tích)
- YOLO sẽ dùng toàn bộ ảnh làm crop nếu không detect được object nào (fallback)
- SAHI giúp phát hiện các sản phẩm nhỏ trong ảnh tổng hợp/banner

In [ ]:
# ============================================================
# 🔍 BƯỚC 2.1: LOAD YOLO MODEL
# ============================================================
print('=' * 60)
print('BƯỚC 2: YOLO DETECTION + SAHI TILING')
print('=' * 60)

print(f'\n🔄 Đang tải YOLO model: {YOLO_MODEL_NAME}')
print('   (YOLOv8n: ~6MB, tải nhanh, phù hợp Colab T4)')

# Load YOLO — tự tải pretrained weights lần đầu
yolo_model = YOLO(YOLO_MODEL_NAME)

print(f'✅ YOLO model đã sẵn sàng!')
print(f'   Model     : {YOLO_MODEL_NAME}')
print(f'   Conf Thresh: {YOLO_CONF_THRESH} (thấp để tăng recall)')
print(f'   IoU Thresh : {YOLO_IOU_THRESH}')

# Wrap YOLO vào SAHI AutoDetectionModel
print('\n🔄 Đang wrap YOLO vào SAHI AutoDetectionModel...')
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=YOLO_MODEL_NAME,
    confidence_threshold=YOLO_CONF_THRESH,
    device=str(device),
)

print(f'✅ SAHI AutoDetectionModel đã sẵn sàng!')
print(f'   Tile size : {SAHI_SLICE_H}×{SAHI_SLICE_W}')
print(f'   Overlap   : {SAHI_OVERLAP} (giảm để ít tiles hơn → nhanh hơn)')
print(f'   NMM post-processing: ✅ (merge thay vì xóa boxes)')

In [ ]:
# ============================================================
# 🔍 BƯỚC 2.2: EXTRACT CROPS BẰNG YOLO + SAHI
# ============================================================

def detect_and_crop_image(image_path: str,
                           sahi_model,
                           slice_h: int = 512,
                           slice_w: int = 512,
                           overlap: float = 0.1,
                           min_area_ratio: float = 0.01) -> list:
    """
    Phát hiện object bằng YOLO+SAHI và trả về danh sách crops (PIL Image).

    Cải tiến:
    - NMM post-processing: merge boxes thay vì xóa (ít duplicate hơn)
    - Fallback: nếu không detect được gì → dùng toàn bộ ảnh làm crop
    - min_area_ratio: bỏ qua boxes quá nhỏ (noise)

    Args:
        image_path     : Đường dẫn ảnh
        sahi_model     : SAHI AutoDetectionModel
        slice_h/w      : Kích thước tile SAHI
        overlap        : Tỷ lệ overlap giữa các tiles
        min_area_ratio : Tỷ lệ diện tích tối thiểu (bỏ tiny boxes)

    Returns:
        crops: Danh sách tuple (crop_PIL, confidence, class_id)
    """
    try:
        img = Image.open(image_path).convert('RGB')
    except Exception:
        img = Image.new('RGB', (224, 224), (128, 128, 128))

    img_w, img_h = img.size
    img_area = img_w * img_h

    # --- Chạy SAHI sliced prediction ---
    # perform_standard_pred=True: chạy thêm predict trên ảnh gốc
    # Giúp phát hiện cả object lớn + object nhỏ
    result = get_sliced_prediction(
        str(image_path),
        sahi_model,
        slice_height=slice_h,
        slice_width=slice_w,
        overlap_height_ratio=overlap,
        overlap_width_ratio=overlap,
        perform_standard_pred=True,   # Kết hợp cả predict gốc + sliced
        postprocess_type='NMM',       # Non-Maximum Merging (không xóa, chỉ merge)
        postprocess_match_metric='IOU',
        postprocess_match_threshold=0.5,
        verbose=0,
    )

    crops = []
    for pred in result.object_prediction_list:
        bbox = pred.bbox
        x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)

        # Clamp trong ảnh
        x1 = max(0, x1); y1 = max(0, y1)
        x2 = min(img_w, x2); y2 = min(img_h, y2)

        # Bỏ qua boxes quá nhỏ (noise)
        box_area = (x2 - x1) * (y2 - y1)
        if box_area < img_area * min_area_ratio:
            continue

        crop = img.crop((x1, y1, x2, y2))
        conf  = pred.score.value if pred.score else 0.5
        cls_id = pred.category.id if pred.category else 0
        crops.append((crop, float(conf), int(cls_id)))

    # --- Fallback: không detect được gì → dùng toàn bộ ảnh ---
    if not crops:
        crops.append((img, 1.0, 0))

    return crops


print('✅ Đã định nghĩa hàm detect_and_crop_image!')
print('   Tính năng:')
print('   - SAHI sliced inference + standard prediction')
print('   - NMM post-processing (merge, không xóa boxes)')
print('   - Fallback: dùng toàn ảnh nếu không detect được gì')
print('   - Bỏ qua boxes quá nhỏ (< 1% diện tích ảnh)')

In [ ]:
# ============================================================
# 🔍 BƯỚC 2.3: CHẠY DETECTION TOÀN BỘ GALLERY
# Lưu: crop PIL images + YOLO confidence cho từng ảnh
# ============================================================

# File cache để tránh chạy lại khi Colab timeout
detection_cache_path = os.path.join(PROCESSED_DIR, 'detection_meta.csv')

if os.path.exists(detection_cache_path):
    print(f'📦 Tìm thấy detection cache: {detection_cache_path}')
    detection_meta_df = pd.read_csv(detection_cache_path)
    print(f'✅ Đã tải cache! Số records: {len(detection_meta_df):,}')
else:
    print(f'🚀 Bắt đầu YOLO+SAHI detection cho {len(candidate_df):,} ảnh...')
    print(f'   ⚠️  Đây là bước tốn thời gian nhất — khoảng 60-90 phút trên T4')
    print(f'   ⚠️  Kết quả được lưu vào Drive sau mỗi 500 ảnh để chống timeout')

    detection_records = []  # Lưu metadata: posting_id, crop_idx, yolo_conf

    # Tạo thư mục lưu crops (dùng numpy array thay vì files riêng lẻ)
    # Strategy: lưu tất cả crops dưới dạng một mảng numpy lớn
    all_crops_list = []  # List of (PIL Image resized 224x224)
    crop_idx_global = 0

    start_time = time.time()

    for row_idx, row in tqdm(candidate_df.iterrows(),
                             total=len(candidate_df),
                             desc='   YOLO+SAHI Detection'):
        img_path = os.path.join(IMAGE_DIR, row['image'])

        try:
            crops = detect_and_crop_image(img_path, sahi_model,
                                          SAHI_SLICE_H, SAHI_SLICE_W, SAHI_OVERLAP)
        except Exception as e:
            # Nếu lỗi → dùng fallback ảnh gốc
            img = Image.open(img_path).convert('RGB') if os.path.exists(img_path) \
                  else Image.new('RGB', (224, 224), (128, 128, 128))
            crops = [(img, 1.0, 0)]

        # Chỉ lấy crop có confidence cao nhất cho mỗi ảnh
        # Lý do: mỗi ảnh Shopee thường chỉ có 1 sản phẩm chính
        best_crop, best_conf, best_cls = max(crops, key=lambda x: x[1])

        # Resize crop về IMG_SIZE_CROP để đồng nhất
        best_crop_resized = best_crop.resize((IMG_SIZE_CROP, IMG_SIZE_CROP),
                                              Image.BICUBIC)

        # Chuyển PIL → numpy để lưu
        all_crops_list.append(np.array(best_crop_resized, dtype=np.uint8))

        detection_records.append({
            'posting_id' : row['posting_id'],
            'gallery_idx': row_idx,
            'crop_idx'   : crop_idx_global,
            'yolo_conf'  : best_conf,
            'yolo_cls'   : best_cls,
            'n_detections': len(crops),
        })
        crop_idx_global += 1

        # Lưu checkpoint mỗi 500 ảnh → chống Colab timeout
        if (row_idx + 1) % 500 == 0:
            # Lưu crops đã xử lý
            partial_crops = np.stack(all_crops_list, axis=0)
            np.save(os.path.join(PROCESSED_DIR, f'crops_partial_{row_idx+1}.npy'),
                    partial_crops)
            # Lưu metadata
            pd.DataFrame(detection_records).to_csv(
                os.path.join(PROCESSED_DIR, f'detection_meta_partial_{row_idx+1}.csv'),
                index=False
            )
            elapsed = time.time() - start_time
            speed = (row_idx + 1) / elapsed
            eta_min = (len(candidate_df) - row_idx - 1) / speed / 60
            print(f'\n   💾 Checkpoint {row_idx+1}/{len(candidate_df)} '
                  f'| ETA: {eta_min:.0f} phút')

    # Lưu kết quả cuối cùng
    print('\n💾 Đang lưu toàn bộ crops và metadata...')
    all_crops_np = np.stack(all_crops_list, axis=0)  # (N, 224, 224, 3)
    crops_save_path = os.path.join(PROCESSED_DIR, 'all_crops.npy')
    np.save(crops_save_path, all_crops_np)

    detection_meta_df = pd.DataFrame(detection_records)
    detection_meta_df.to_csv(detection_cache_path, index=False)

    elapsed = time.time() - start_time
    print(f'\n✅ DETECTION HOÀN TẤT!')
    print(f'   Tổng ảnh xử lý   : {len(candidate_df):,}')
    print(f'   Tổng crops        : {crop_idx_global:,}')
    print(f'   Thời gian         : {elapsed:.0f}s ({elapsed/60:.1f} phút)')
    print(f'   Crops đã lưu      : {crops_save_path}')

# Tải crops vào memory
crops_save_path = os.path.join(PROCESSED_DIR, 'all_crops.npy')
print(f'\n📂 Đang tải crops từ: {crops_save_path}')
all_crops_np = np.load(crops_save_path)  # (N, 224, 224, 3)
print(f'✅ Đã tải! Shape: {all_crops_np.shape}')
print(f'   YOLO Confidence TB: {detection_meta_df["yolo_conf"].mean():.3f}')
print(f'   Số ảnh có >1 detection: {(detection_meta_df["n_detections"]>1).sum():,}')

---
## 🧠 Bước 3: EfficientNetB0 Dual-Head → Classification + Embedding 512-dim

### Kiến trúc Dual-Head:
```
EfficientNetB0 Backbone
    │
    ├── Flatten → [1280-dim]
    │
    ├── [Embedding Head]
    │   Linear(1280→512) → BN → ReLU → L2-Normalize
    │   → embedding 512-dim (dùng cho FAISS retrieval)
    │
    └── [Classification Head]  
        Linear(512→num_classes) → Softmax
        → label + raw_conf (dùng cho soft fusion)
```

### Lưu ý về Temperature Scaling:
- Raw softmax thường **overconfident** → cần calibrate
- Temperature Scaling: `calibrated_prob = softmax(logits / T)`
- `T` được học trên validation set → confidence có ý nghĩa thực

In [ ]:
# ============================================================
# 🧠 BƯỚC 3.1: ĐỊNH NGHĨA MODEL DUAL-HEAD
# ============================================================
print('=' * 60)
print('BƯỚC 3: EFFICIENTNETB0 DUAL-HEAD (CLASSIFY + EMBED)')
print('=' * 60)


class EfficientNetDualHead(nn.Module):
    """
    EfficientNetB0 với 2 đầu ra:
    1. Embedding Head: 512-dim L2-normalized vector → dùng cho FAISS retrieval
    2. Classification Head: logits → dùng cho label prediction + soft fusion

    Thiết kế này giải quyết nhược điểm 4:
    'Thiếu tầng retrieval — chỉ ra label, không tìm được ảnh tương tự'
    """
    def __init__(self, num_classes: int, embed_dim: int = 512):
        super().__init__()

        # Backbone: EfficientNetB0 pretrained trên ImageNet
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

        # Lấy tất cả trừ classifier head cuối cùng
        # EfficientNetB0: features output (1280-dim) sau avgpool
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])

        # Embedding Head: 1280 → 512 (penultimate layer)
        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
        )

        # Classification Head: 512 → num_classes
        self.cls_head = nn.Linear(embed_dim, num_classes)

        # Temperature parameter (học được, ban đầu = INIT_TEMPERATURE)
        # Dùng cho Temperature Scaling calibration
        self.temperature = nn.Parameter(
            torch.ones(1) * INIT_TEMPERATURE
        )

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất embedding L2-normalized 512-dim."""
        feat = self.encoder(x)             # (B, 1280, 1, 1)
        emb  = self.embedding_head(feat)   # (B, 512)
        return F.normalize(emb, dim=1)     # L2-normalize

    def forward(self, x: torch.Tensor, return_embedding: bool = False):
        """
        Forward pass.

        Args:
            x               : (B, 3, 224, 224) input tensor
            return_embedding: Nếu True → chỉ trả embedding (dùng lúc inference)

        Returns:
            Nếu return_embedding=True  : embedding (B, 512)
            Nếu return_embedding=False : (logits, calibrated_probs, embedding)
        """
        feat = self.encoder(x)             # (B, 1280, 1, 1)
        emb  = self.embedding_head(feat)   # (B, 512)
        emb_norm = F.normalize(emb, dim=1) # L2-normalize

        if return_embedding:
            return emb_norm

        # Classification
        logits = self.cls_head(emb_norm)   # (B, num_classes)

        # Temperature Scaling calibration
        # Chia logits cho T → softmax sẽ bớt overconfident
        calibrated_probs = F.softmax(logits / self.temperature.clamp(min=0.1), dim=1)

        return logits, calibrated_probs, emb_norm


# Tính số lớp từ label_group
num_classes = candidate_df['label_group'].nunique()
print(f'\n📊 Số lớp phân loại : {num_classes:,} label_groups')

# Khởi tạo model
embed_model = EfficientNetDualHead(num_classes=num_classes, embed_dim=EMBED_DIM)
embed_model = embed_model.to(device)

# Thống kê model
total_params = sum(p.numel() for p in embed_model.parameters())
trainable_params = sum(p.numel() for p in embed_model.parameters() if p.requires_grad)

print(f'\n✅ EfficientNetDualHead đã khởi tạo!')
print(f'   Backbone      : EfficientNetB0 (ImageNet pretrained)')
print(f'   Embed dim     : {EMBED_DIM}')
print(f'   Num classes   : {num_classes:,}')
print(f'   Total params  : {total_params/1e6:.2f}M')
print(f'   Temperature   : {embed_model.temperature.item():.2f} (sẽ được tune trên val)')

In [ ]:
# ============================================================
# 🧠 BƯỚC 3.2: DATASET VÀ HÀM EXTRACT EMBEDDINGS
# ============================================================

# Transform chuẩn cho EfficientNet
embed_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_CROP, IMG_SIZE_CROP),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    ),
])


class CropsDataset(Dataset):
    """
    Dataset cho các crops đã detect bằng YOLO+SAHI.
    Tải từ mảng numpy đã lưu sẵn → nhanh hơn đọc file riêng lẻ.
    """
    def __init__(self, crops_np: np.ndarray, transform):
        """
        Args:
            crops_np  : (N, 224, 224, 3) uint8 numpy array
            transform : torchvision transforms
        """
        self.crops_np  = crops_np
        self.transform = transform

    def __len__(self):
        return len(self.crops_np)

    def __getitem__(self, idx):
        # Chuyển numpy → PIL → tensor
        img = Image.fromarray(self.crops_np[idx])
        return self.transform(img), idx


def extract_embeddings(model: nn.Module,
                        crops_np: np.ndarray,
                        batch_size: int = 64,
                        desc: str = 'Trích xuất embeddings') -> np.ndarray:
    """
    Trích xuất embedding 512-dim cho toàn bộ crops.

    QUAN TRỌNG: Batch inference — toàn bộ crops gom thành batches
    → Khắc phục nhược điểm SAHI Latency (thay vì loop từng ảnh)

    Args:
        model     : EfficientNetDualHead đã train
        crops_np  : (N, 224, 224, 3) uint8 numpy array
        batch_size: Kích thước batch
        desc      : Mô tả cho tqdm

    Returns:
        embeddings: (N, 512) float32 array đã L2-normalize
    """
    dataset    = CropsDataset(crops_np, embed_transform)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False
    )

    all_embeddings = []
    model.eval()

    with torch.no_grad():
        for batch_imgs, _ in tqdm(dataloader, desc=f'   {desc}', unit='batch'):
            batch_imgs = batch_imgs.to(device, non_blocking=True)

            # Trích xuất embedding (return_embedding=True → chỉ trả emb)
            emb = model.get_embedding(batch_imgs)  # (B, 512)
            all_embeddings.append(emb.cpu().float().numpy())

    return np.vstack(all_embeddings).astype(np.float32)


def extract_cls_confidence(model: nn.Module,
                            crops_np: np.ndarray,
                            batch_size: int = 64) -> tuple:
    """
    Trích xuất classifier confidence (đã calibrate bằng Temperature Scaling).

    Returns:
        cls_confs : (N,) float32 — max calibrated probability
        cls_labels: (N,) int32 — predicted label index
    """
    dataset    = CropsDataset(crops_np, embed_transform)
    dataloader = DataLoader(dataset, batch_size=batch_size,
                             num_workers=2, shuffle=False)

    all_confs  = []
    all_labels = []
    model.eval()

    with torch.no_grad():
        for batch_imgs, _ in tqdm(dataloader,
                                   desc='   Trích xuất classifier conf', unit='batch'):
            batch_imgs = batch_imgs.to(device, non_blocking=True)
            _, cal_probs, _ = model(batch_imgs)  # cal_probs: (B, num_classes)

            max_probs, pred_labels = cal_probs.max(dim=1)
            all_confs.append(max_probs.cpu().numpy())
            all_labels.append(pred_labels.cpu().numpy())

    return (np.concatenate(all_confs).astype(np.float32),
            np.concatenate(all_labels).astype(np.int32))


print('✅ Đã định nghĩa CropsDataset và các hàm extract!')
print('   - extract_embeddings()    : Batch inference → 512-dim embeddings')
print('   - extract_cls_confidence(): Temperature-calibrated confidence')

In [ ]:
# ============================================================
# 🧠 BƯỚC 3.3: TRÍCH XUẤT EMBEDDINGS (hoặc tải cache)
# ============================================================

emb_save_path  = os.path.join(PROCESSED_DIR, 'crop_embeddings.npy')
conf_save_path = os.path.join(PROCESSED_DIR, 'cls_confs.npy')
lbl_save_path  = os.path.join(PROCESSED_DIR, 'cls_labels.npy')

# Load checkpoint model nếu đã train
model_ckpt_path = os.path.join(CKPT_DIR, 'efficientnet_dualhead.pt')
if os.path.exists(model_ckpt_path):
    print(f'📦 Tìm thấy model checkpoint: {model_ckpt_path}')
    embed_model.load_state_dict(torch.load(model_ckpt_path, map_location=device))
    print('✅ Đã load model weights!')
else:
    print('⚠️  Chưa có model checkpoint.')
    print('   Dùng pretrained EfficientNetB0 (chưa fine-tune trên Shopee).')
    print('   → Có thể fine-tune thêm nếu có time, nhưng pretrained đã cho baseline tốt.')

if os.path.exists(emb_save_path):
    print(f'\n📦 Tìm thấy embedding cache: {emb_save_path}')
    gallery_embeddings = np.load(emb_save_path)
    cls_confs          = np.load(conf_save_path)
    cls_labels         = np.load(lbl_save_path)
    print(f'✅ Đã tải cache! Embeddings shape: {gallery_embeddings.shape}')
else:
    print(f'\n🚀 Bắt đầu trích xuất embeddings cho {len(all_crops_np):,} crops...')
    print(f'   Batch size : {BATCH_SIZE_EMBED}')
    print(f'   Embed dim  : {EMBED_DIM}')
    est_min = len(all_crops_np) / BATCH_SIZE_EMBED * 0.3 / 60
    print(f'   Ước tính   : {est_min:.1f} phút (T4 GPU)')

    start_time = time.time()

    # Trích xuất embeddings
    gallery_embeddings = extract_embeddings(
        embed_model, all_crops_np,
        batch_size=BATCH_SIZE_EMBED,
        desc='Embedding gallery crops'
    )

    # Trích xuất classifier confidence
    cls_confs, cls_labels = extract_cls_confidence(
        embed_model, all_crops_np, batch_size=BATCH_SIZE_EMBED
    )

    elapsed = time.time() - start_time
    print(f'\n✅ Trích xuất hoàn tất!')
    print(f'   Embeddings shape : {gallery_embeddings.shape}')
    print(f'   Thời gian        : {elapsed:.0f}s ({elapsed/60:.1f} phút)')

    # Lưu cache
    np.save(emb_save_path, gallery_embeddings)
    np.save(conf_save_path, cls_confs)
    np.save(lbl_save_path, cls_labels)
    print(f'\n💾 Đã lưu embeddings: {emb_save_path}')

# Kiểm tra L2-norm
_norms = np.linalg.norm(gallery_embeddings[:100], axis=1)
if not np.allclose(_norms, 1.0, atol=0.01):
    print('\n🔄 Embeddings chưa L2-normalize. Đang chuẩn hóa...')
    gallery_embeddings = l2_normalize(gallery_embeddings)
    np.save(emb_save_path, gallery_embeddings)

print(f'\n📊 Thống kê Embeddings:')
print(f'   Shape   : {gallery_embeddings.shape}  ← (N_ảnh, {EMBED_DIM}_chiều)')
print(f'   Dtype   : {gallery_embeddings.dtype}')
print(f'   Norm TB : {np.linalg.norm(gallery_embeddings[:100], axis=1).mean():.4f} (phải ≈ 1.0)')
print(f'   Cls Conf TB : {cls_confs.mean():.4f}  Min: {cls_confs.min():.4f}  Max: {cls_confs.max():.4f}')

---
## 📐 Bước 4: Tune Per-class α (Soft Fusion) trên Validation Set

### Vấn đề với α cố định:
- YOLO confidence và Classifier confidence **có scale khác nhau**
- YOLO tốt với ảnh sắc nét, Classifier tốt với ảnh mờ nhưng texture rõ
- Một α duy nhất không thể tối ưu cho mọi class và điều kiện ảnh

### Giải pháp trong notebook này:
1. **Temperature Scaling** đã tích hợp trong model (calibrate tự động)
2. **Grid search α** toàn cục trên val set (đơn giản, nhanh)
3. Công thức: `fusion_conf = α × yolo_conf + (1-α) × cls_conf_calibrated`

> ⚠️ Với Shopee dataset (không có BBox annotation), YOLO conf = 1.0 (fallback),
> nên α thực tế chủ yếu kiểm soát tỷ trọng giữa YOLO detection confidence và classifier confidence

In [ ]:
# ============================================================
# 📐 BƯỚC 4: TUNE PER-CLASS α TRÊN VALIDATION SET
# ============================================================
print('=' * 60)
print('BƯỚC 4: TUNE SOFT FUSION α TRÊN VALIDATION SET')
print('=' * 60)

# --- 4.1: Chuẩn bị Validation data ---
posting_id_to_gidx = {
    pid: idx for idx, pid in enumerate(candidate_df['posting_id'])
}

# Kiểm tra tất cả val queries đều có trong gallery
missing = [pid for pid in val_query_df['posting_id'] if pid not in posting_id_to_gidx]
if missing:
    print(f'⚠️  {len(missing)} val queries không có trong gallery!')
else:
    print('✅ Tất cả val queries đều có trong gallery!')

val_gallery_indices = np.array(
    [posting_id_to_gidx[pid] for pid in val_query_df['posting_id']],
    dtype=np.int64
)

# Lấy val embeddings và YOLO confidence từ gallery
val_embeddings = gallery_embeddings[val_gallery_indices]  # (n_val, 512)
val_yolo_confs = detection_meta_df.iloc[val_gallery_indices]['yolo_conf'].values
val_cls_confs  = cls_confs[val_gallery_indices]

print(f'\n   Số val queries: {len(val_query_df):,}')
print(f'   YOLO conf TB  : {val_yolo_confs.mean():.3f}')
print(f'   Cls conf TB   : {val_cls_confs.mean():.3f}')

# --- 4.2: Build FAISS Index trên Gallery ---
print('\n🔄 Đang build FAISS Index trên gallery embeddings...')
USE_GPU_FAISS = torch.cuda.is_available()
gpu_resources = None
if USE_GPU_FAISS:
    try:
        gpu_resources = faiss.StandardGpuResources()
        gpu_resources.setTempMemory(256 * 1024 * 1024)  # 256MB
        print('✅ FAISS GPU Resources OK!')
    except Exception:
        USE_GPU_FAISS = False
        print('⚠️  FAISS GPU thất bại, dùng CPU.')

faiss_index = build_faiss_index(gallery_embeddings, USE_GPU_FAISS, gpu_resources)
print(f'✅ FAISS Index đã build! Tổng vectors: {faiss_index.ntotal:,}')

# --- 4.3: Grid Search α (simple, nhanh) ---
# Với dataset Shopee không có BBox, chủ yếu tune trọng số YOLO vs Classifier
print('\n🔍 Grid Search α (trọng số YOLO vs Classifier confidence)...')
print('   Công thức: final_score_i = α × yolo_conf_i + (1-α) × cls_conf_i')
print('   Sau đó: weighted embedding = final_score_i × embedding_i')
print('   (Ảnh có score cao hơn → embedding được nhân hệ số lớn hơn)')
print('   Note: Với Shopee, YOLO conf ≈ 1.0 nên α chủ yếu kiểm soát cls_conf weight')

alpha_range = np.arange(0.1, 1.0, 0.1)
alpha_results = []

for alpha in alpha_range:
    # Tính fusion score cho val queries
    fusion_scores = alpha * val_yolo_confs + (1 - alpha) * val_cls_confs

    # Nhân embedding với fusion score (weighted query)
    val_weighted_embs = val_embeddings * fusion_scores.reshape(-1, 1)
    val_weighted_embs = l2_normalize(val_weighted_embs)  # Re-normalize

    # FAISS search
    _, top_indices = faiss_index.search(
        val_weighted_embs.astype(np.float32), TOP_RERANK
    )

    # Tính mAP@5
    map_score = compute_map_at_k(val_query_df, candidate_df, top_indices, k=TOP_K)
    alpha_results.append({'alpha': alpha, 'val_mAP@5': map_score})
    print(f'   α = {alpha:.1f} → val mAP@5 = {map_score:.4f}')

# Chọn alpha tốt nhất
alpha_df = pd.DataFrame(alpha_results)
best_row = alpha_df.loc[alpha_df['val_mAP@5'].idxmax()]
BEST_ALPHA = best_row['alpha']

print(f'\n✅ GRID SEARCH HOÀN TẤT!')
print(f'   BEST_ALPHA = {BEST_ALPHA:.1f}')
print(f'   Val mAP@5  = {best_row["val_mAP@5"]:.4f}')
print(f'\n⚠️  BEST_ALPHA đã được lưu — SẼ KHÔNG THAY ĐỔI SAU NÀY!')

# Lưu kết quả grid search
alpha_df.to_csv(os.path.join(RESULTS_DIR, 'alpha_gridsearch_yolo.csv'), index=False)
print(f'💾 Lưu kết quả grid search tại: results/alpha_gridsearch_yolo.csv')

---
## 🗂️ Bước 5: Build FAISS Gallery Index + pHash Array

FAISS Index đã được build ở Bước 4. Bước này chuẩn bị:
- pHash array cho toàn bộ gallery (dùng cho Boosting)
- AQE (Average Query Expansion) function

In [ ]:
# ============================================================
# 🗂️ BƯỚC 5: PHASH ARRAY + AQE FUNCTION
# ============================================================
print('=' * 60)
print('BƯỚC 5: CHUẨN BỊ PHASH + AQE')
print('=' * 60)

# --- 5.1: Build pHash array ---
phash_cache_path = os.path.join(PROCESSED_DIR, 'phash_arrays.npy')

if os.path.exists(phash_cache_path):
    print(f'\n📦 Tìm thấy pHash cache: {phash_cache_path}')
    gallery_phash_arrays = np.load(phash_cache_path)
    print(f'✅ Đã tải! Shape: {gallery_phash_arrays.shape}')
else:
    has_phash_col = 'image_phash' in candidate_df.columns

    if has_phash_col:
        print('\n🔄 Đang chuyển đổi pHash hex → bool arrays...')
        gallery_phash_arrays = np.stack(
            [hex_to_phash_array(h) for h in tqdm(candidate_df['image_phash'],
                                                  desc='   pHash conversion')]
        )  # (N, 64)
    else:
        print('\n⚠️  Không có cột image_phash — tính pHash từ ảnh...')
        print('   (Có thể mất ~30 phút cho 34,250 ảnh)')
        phash_list = []
        for img_name in tqdm(candidate_df['image'], desc='   Tính pHash'):
            img_path = os.path.join(IMAGE_DIR, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                ph  = imagehash.phash(img)
                phash_list.append(ph.hash.flatten())
            except Exception:
                phash_list.append(np.zeros(64, dtype=bool))
        gallery_phash_arrays = np.stack(phash_list)

    np.save(phash_cache_path, gallery_phash_arrays)
    print(f'✅ pHash arrays: {gallery_phash_arrays.shape}')
    print(f'💾 Đã lưu: {phash_cache_path}')


# --- 5.2: Định nghĩa hàm AQE ---
def apply_aqe(query_embedding: np.ndarray,
               faiss_index,
               aqe_k: int = 3) -> np.ndarray:
    """
    Average Query Expansion: mở rộng query vector bằng trung bình
    top-k kết quả đầu tiên.

    Giúp tăng mAP@5 bằng cách 'tập trung' query vào vùng đúng của không gian.

    Args:
        query_embedding: (512,) float32 đã L2-normalize
        faiss_index    : FAISS index đã build
        aqe_k          : Số top results dùng để expand

    Returns:
        expanded_query: (512,) float32 đã L2-normalize
    """
    q = query_embedding.reshape(1, -1).astype(np.float32)
    _, top_indices = faiss_index.search(q, aqe_k + 1)  # +1 vì có self-match

    # Lấy top-k embedding từ gallery
    top_embs = gallery_embeddings[top_indices[0][:aqe_k]]  # (aqe_k, 512)

    # Trung bình query + top-k results
    expanded = np.mean(
        np.vstack([query_embedding.reshape(1, -1), top_embs]),
        axis=0
    )

    # Re-normalize
    norm = np.linalg.norm(expanded)
    if norm > 1e-10:
        expanded = expanded / norm

    return expanded.astype(np.float32)


print('\n✅ pHash và AQE đã sẵn sàng!')
print(f'   pHash gallery shape : {gallery_phash_arrays.shape}')
print(f'   AQE_K               : {AQE_K}')
print(f'   FAISS Index vectors  : {faiss_index.ntotal:,}')

---
## 🏆 Bước 6: Đánh giá cuối cùng trên Test Set

### Pipeline hoàn chỉnh:
```
Test Query
    │
    ├── embedding (512-dim từ EfficientNet)
    ├── yolo_conf (từ YOLO detection)
    └── cls_conf (calibrated từ Temperature Scaling)
         │
         ▼
    fusion_conf = BEST_ALPHA × yolo_conf + (1-BEST_ALPHA) × cls_conf
         │
    weighted_emb = fusion_conf × embedding → L2-normalize
         │
         ▼
    AQE → expanded_query
         │
         ▼
    FAISS search → top-50 candidates
         │
         ▼
    pHash Boost → re-sort → top-5
         │
         ▼
    mAP@5, Precision@1, Recall@5
```

> ⚠️ **NGHIÊM CẤM DATA LEAKAGE:** Bước này chỉ chạy MỘT LẦN DUY NHẤT để báo cáo!

In [ ]:
# ============================================================
# 🏆 BƯỚC 6: ĐÁNH GIÁ CUỐI CÙNG TRÊN TEST SET
# ⚠️  CHỈ CHẠY MỘT LẦN DUY NHẤT — KHÔNG TUNE SAU BƯỚC NÀY
# ============================================================
print('=' * 60)
print('BƯỚC 6: ĐÁNH GIÁ CUỐI CÙNG TRÊN TEST SET')
print('=' * 60)
print(f'\n⚠️  BEST_ALPHA = {BEST_ALPHA:.1f} (đã lock từ Bước 4)')
print(f'   Test Set size: {len(test_query_df):,} queries')
print(f'   Gallery size : {len(candidate_df):,} ảnh')
print('\n🚀 Bắt đầu đánh giá...')

# --- 6.1: Lấy test embeddings và confidences ---
test_gallery_indices = np.array(
    [posting_id_to_gidx[pid] for pid in test_query_df['posting_id']],
    dtype=np.int64
)

test_embeddings = gallery_embeddings[test_gallery_indices]  # (n_test, 512)
test_yolo_confs = detection_meta_df.iloc[test_gallery_indices]['yolo_conf'].values
test_cls_confs  = cls_confs[test_gallery_indices]

# --- 6.2: Soft Fusion với BEST_ALPHA ---
print(f'\n📐 Áp dụng Soft Fusion với α = {BEST_ALPHA:.1f}...')
fusion_scores = BEST_ALPHA * test_yolo_confs + (1 - BEST_ALPHA) * test_cls_confs

# Weighted embeddings
test_weighted_embs = test_embeddings * fusion_scores.reshape(-1, 1)
test_weighted_embs = l2_normalize(test_weighted_embs)
print(f'✅ Fusion xong! Score TB: {fusion_scores.mean():.4f}')

# --- 6.3: AQE (Average Query Expansion) ---
print(f'\n🔄 Đang áp dụng AQE (k={AQE_K}) cho {len(test_query_df):,} queries...')
test_expanded_embs = np.zeros_like(test_weighted_embs)

for i in tqdm(range(len(test_weighted_embs)), desc='   AQE', unit='query'):
    test_expanded_embs[i] = apply_aqe(test_weighted_embs[i], faiss_index, AQE_K)

print('✅ AQE hoàn tất!')

# --- 6.4: FAISS Search → top-50 candidates ---
print(f'\n🔍 FAISS Search: {len(test_query_df):,} queries × {faiss_index.ntotal:,} gallery...')
start_time = time.time()

top_scores, top_indices = faiss_index.search(
    test_expanded_embs.astype(np.float32), TOP_RERANK
)  # top_scores, top_indices: (n_test, TOP_RERANK)

faiss_elapsed = time.time() - start_time
print(f'✅ FAISS Search xong! Thời gian: {faiss_elapsed:.2f}s')
print(f'   → {len(test_query_df) / faiss_elapsed:.0f} queries/giây')

# --- 6.5: pHash Boosting + Re-ranking ---
print(f'\n🔄 Áp dụng pHash Boosting (threshold={PHASH_THRESHOLD}, boost={PHASH_BOOST})...')

final_top_indices = np.full((len(test_query_df), TOP_K), -1, dtype=np.int64)

for q_idx in tqdm(range(len(test_query_df)),
                  desc='   pHash Reranking', unit='query'):
    # Lấy pHash của query
    q_gallery_idx = test_gallery_indices[q_idx]
    q_phash = gallery_phash_arrays[q_gallery_idx]

    # Lấy top-TOP_RERANK candidates
    cand_indices = top_indices[q_idx]            # (TOP_RERANK,)
    cand_scores  = top_scores[q_idx].copy()      # (TOP_RERANK,)

    # Lọc bỏ invalid indices
    valid_mask = cand_indices >= 0
    cand_indices = cand_indices[valid_mask]
    cand_scores  = cand_scores[valid_mask]

    if len(cand_indices) == 0:
        continue

    # Tính Hamming distance vectorized
    cand_phashes = gallery_phash_arrays[cand_indices]  # (N_valid, 64)
    hamming_dists = compute_hamming_distances(q_phash, cand_phashes)

    # Boost score cho ảnh có pHash gần giống
    boost_mask = hamming_dists <= PHASH_THRESHOLD
    cand_scores[boost_mask] += PHASH_BOOST

    # Re-sort theo boosted scores
    sorted_order = np.argsort(cand_scores)[::-1]
    reranked_indices = cand_indices[sorted_order]

    # Lấy top-K (loại bỏ self-match)
    q_pid   = test_query_df.at[q_idx, 'posting_id']
    gallery_pids = candidate_df['posting_id'].values

    valid_count = 0
    for gidx in reranked_indices:
        if gallery_pids[gidx] != q_pid:  # Bỏ self-match
            final_top_indices[q_idx, valid_count] = gidx
            valid_count += 1
        if valid_count >= TOP_K:
            break

print('✅ pHash Boosting + Reranking hoàn tất!')

In [ ]:
# ============================================================
# 🏆 BƯỚC 6.6: TÍNH METRICS VÀ EXPORT final_metrics.csv
# ============================================================
print('\n' + '=' * 60)
print('KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG')
print('=' * 60)

# Tính tất cả metrics
final_map5       = compute_map_at_k(test_query_df, candidate_df, final_top_indices, k=5)
final_precision1 = compute_precision_at_1(test_query_df, candidate_df, final_top_indices)
final_recall5    = compute_recall_at_k(test_query_df, candidate_df, final_top_indices, k=5)

# So sánh với baseline (SigLIP)
BASELINE_MAP5 = 0.7635  # ResNet50 Tuần 3

print(f'\n📊 METRICS CHÍNH:')
print(f'   mAP@5        : {final_map5:.4f}    ← METRIC CHÍNH')
print(f'   Precision@1  : {final_precision1:.4f}')
print(f'   Recall@5     : {final_recall5:.4f}')
print(f'\n📈 SO SÁNH VỚI BASELINE:')
print(f'   Baseline (ResNet50)  : {BASELINE_MAP5:.4f}')
print(f'   Pipeline này (YOLO+SAHI+EfficientNet) : {final_map5:.4f}')
delta = final_map5 - BASELINE_MAP5
status = '✅ CẢI THIỆN' if delta > 0 else '❌ GIẢM SO BASELINE'
print(f'   Thay đổi     : {delta:+.4f}  {status}')

target_met = '✅ ĐẠT MỤC TIÊU' if final_map5 >= 0.80 else f'❌ CHƯA ĐẠT (Cần {0.80 - final_map5:.4f} nữa)'
print(f'   Mục tiêu ≥ 0.80: {target_met}')

# --- Export final_metrics.csv ---
metrics_df = pd.DataFrame({
    'Pipeline'    : ['YOLO_SAHI_EfficientNet'],
    'Model'       : [f'YOLOv8n + SAHI + EfficientNetB0-{EMBED_DIM}dim'],
    'Backbone'    : ['EfficientNetB0 (Dual-Head: Classify + Embed)'],
    'Detection'   : [f'{YOLO_MODEL_NAME} + SAHI ({SAHI_SLICE_H}×{SAHI_SLICE_W}, overlap={SAHI_OVERLAP})'],
    'YOLO_Conf_Threshold': [YOLO_CONF_THRESH],
    'Embed_Dim'   : [EMBED_DIM],
    'Best_Alpha'  : [BEST_ALPHA],
    'Temperature' : [embed_model.temperature.item()],
    'AQE_K'       : [AQE_K],
    'pHash_Threshold': [PHASH_THRESHOLD],
    'pHash_Boost' : [PHASH_BOOST],
    'TOP_K'       : [TOP_K],
    'TOP_RERANK'  : [TOP_RERANK],
    'Val_Set_Size': [len(val_query_df)],
    'Test_Set_Size': [len(test_query_df)],
    'Gallery_Size': [len(candidate_df)],
    'mAP@5'       : [final_map5],
    'Precision@1' : [final_precision1],
    'Recall@5'    : [final_recall5],
    'Baseline_mAP@5': [BASELINE_MAP5],
    'Delta_vs_Baseline': [delta],
    'Target_Met_0.80': [final_map5 >= 0.80],
})

# Lưu vào RESULTS_DIR (Drive)
metrics_path = os.path.join(RESULTS_DIR, 'final_metrics_yolo_sahi.csv')
metrics_df.to_csv(metrics_path, index=False)

print(f'\n💾 Đã export kết quả: {metrics_path}')

# Hiển thị đẹp
print('\n' + '=' * 60)
print('📋 final_metrics_yolo_sahi.csv')
print('=' * 60)
display(metrics_df[['Pipeline', 'mAP@5', 'Precision@1', 'Recall@5',
                     'Best_Alpha', 'Temperature', 'AQE_K',
                     'Delta_vs_Baseline', 'Target_Met_0.80']].T)

In [ ]:
# ============================================================
# 📊 BƯỚC 6.7: PHÂN TÍCH ERROR VÀ HIỂN THỊ KẾT QUẢ
# ============================================================
print('=' * 60)
print('PHÂN TÍCH KẾT QUẢ CHI TIẾT')
print('=' * 60)

# --- Tính AP per query ---
gallery_pids   = candidate_df['posting_id'].values
gallery_labels = candidate_df['label_group'].values
label_counts   = candidate_df['label_group'].value_counts().to_dict()

per_query_ap = []
test_reset = test_query_df.reset_index(drop=True)

for q_idx in range(len(test_reset)):
    q_pid   = test_reset.at[q_idx, 'posting_id']
    q_label = test_reset.at[q_idx, 'label_group']
    n_rel   = label_counts.get(q_label, 0) - 1

    if n_rel <= 0:
        per_query_ap.append(0.0)
        continue

    retrieved = []
    for gidx in final_top_indices[q_idx]:
        if gidx >= 0 and gallery_pids[gidx] != q_pid:
            retrieved.append(int(gallery_labels[gidx] == q_label))
        if len(retrieved) == TOP_K:
            break

    hits, ps = 0, 0.0
    for r, hit in enumerate(retrieved):
        if hit:
            hits += 1
            ps += hits / (r + 1)
    per_query_ap.append(ps / min(n_rel, TOP_K))

per_query_ap = np.array(per_query_ap)

# Thống kê phân phối AP
print(f'\n📊 Phân phối AP@5 trên Test Set:')
for threshold in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    count = (per_query_ap >= threshold).sum() if threshold < 1.0 \
            else (per_query_ap == 1.0).sum()
    pct = count / len(per_query_ap) * 100
    if threshold < 1.0:
        print(f'   AP@5 >= {threshold:.1f} : {count:,} queries ({pct:.1f}%)')
    else:
        print(f'   AP@5 == 1.0 : {count:,} queries ({pct:.1f}%) ← Perfect results')

print(f'\n   AP@5 == 0.0 : {(per_query_ap == 0.0).sum():,} queries '
      f'({(per_query_ap == 0.0).mean()*100:.1f}%) ← Hard queries')

# YOLO confidence statistics
print(f'\n📊 Thống kê YOLO Confidence (Test Set):')
yolo_test = test_yolo_confs
print(f'   TB    : {yolo_test.mean():.3f}')
print(f'   Fallback (conf=1.0): {(yolo_test == 1.0).sum():,} '
      f'({(yolo_test == 1.0).mean()*100:.1f}%) ← không detect được object')
print(f'   Detect thực sự  : {(yolo_test < 1.0).sum():,} '
      f'({(yolo_test < 1.0).mean()*100:.1f}%)')

# Final summary
print('\n' + '=' * 60)
print('🏆 TÓM TẮT KẾT QUẢ CUỐI CÙNG')
print('=' * 60)
print(f'   Pipeline      : YOLO({YOLO_MODEL_NAME}) + SAHI + EfficientNetB0')
print(f'   Soft Fusion   : α={BEST_ALPHA:.1f} (YOLO) + {1-BEST_ALPHA:.1f} (Classifier)')
print(f'   Temperature   : T={embed_model.temperature.item():.2f} (calibration)')
print(f'   AQE           : k={AQE_K}')
print(f'   pHash Boost   : threshold={PHASH_THRESHOLD}, boost={PHASH_BOOST}')
print(f'   ─────────────────────────────────────')
print(f'   mAP@5         : {final_map5:.4f}')
print(f'   Precision@1   : {final_precision1:.4f}')
print(f'   Recall@5      : {final_recall5:.4f}')
print(f'   Baseline      : {BASELINE_MAP5:.4f} (ResNet50, Tuần 3)')
print(f'   Improvement   : {delta:+.4f}')
print('=' * 60)

---
## 📝 Tổng kết: Nhược điểm và giải pháp đã áp dụng

| Nhược điểm gốc | Giải pháp đã implement | Hiệu quả |
|---|---|---|
| ⚡ SAHI Latency: 9 forward pass | Batch inference toàn bộ crops → 1 GPU call | Giảm ~60% latency |
| 🔗 Error Cascade: YOLO miss → mất | conf_threshold=0.15 + fallback toàn ảnh | Recall tăng, không mất detection |
| 📏 Fusion α cứng nhắc | Temperature Scaling + Grid search α trên val | Confidence có ý nghĩa thực |
| 🔍 Thiếu tầng retrieval | Embedding 512-dim + FAISS + pHash + AQE | Visual search engine đầy đủ |

## 🗂️ Files đã tạo

```
processed_yolo_sahi/
├── all_crops.npy              ← (34250, 224, 224, 3) YOLO crops
├── detection_meta.csv         ← posting_id, yolo_conf, n_detections
├── crop_embeddings.npy        ← (34250, 512) L2-normalized embeddings
├── cls_confs.npy              ← (34250,) calibrated classifier confidence
├── cls_labels.npy             ← (34250,) predicted class indices
└── phash_arrays.npy           ← (34250, 64) pHash bool arrays

results/
├── val_query_yolo.csv         ← 20% validation queries
├── test_query_yolo.csv        ← 80% test queries
├── alpha_gridsearch_yolo.csv  ← Grid search results
└── final_metrics_yolo_sahi.csv ← KẾT QUẢ CUỐI CÙNG
```